### Batch Summaries

Imports

In [1]:
import yaml
import pandas as pd
import os
import re

#### Parameters

In [3]:

# Server names
server_dicer2 = 'dicer2'
server_aws1 = 'aws1'
server_aws2 = 'aws2'
server_r2d2 = 'r2d2'

# Columns of lib .csv output
col_label_similarity = 'Similarity_label'
col_label_class = 'class'

# Columns of batch report output
col_input = 'Input Size'
col_output = 'Output Size'
col_reads = 'Reads'

# Paths & file names
path_root = './file/run-summaries/'
path_export = './file/executions-summary.csv'

suffix_summary = '.summary.yml'
suffix_classification = '-viral-eve.csv'


#### EDA

In [45]:
df_lib.columns

Index(['Contigs_ID', 'Similarity_label', 'X15', 'X16', 'X17', 'X18', 'X19',
       'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29',
       'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X.15', 'X.16', 'X.17',
       'X.18', 'X.19', 'X.20', 'X.21', 'X.22', 'X.23', 'X.24', 'X.25', 'X.26',
       'X.27', 'X.28', 'X.29', 'X.30', 'X.31', 'X.32', 'X.33', 'X.34', 'X.35',
       'dens15to18', 'dens20to22', 'dens25to29', 'ratiosi_pi', 'ratio_si',
       'dens18to35', 'class'],
      dtype='object')

In [46]:
df_lib.shape

(54, 51)

In [43]:
df_lib.describe()

,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,...,X.32,X.33,X.34,X.35,dens15to18,dens20to22,dens25to29,ratiosi_pi,ratio_si,dens18to35
count,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,...,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000,54.000000
mean,-0.460346,-0.460346,-0.460346,-0.458855,-0.447591,-0.356837,0.444018,-0.280465,0.115610,0.330328,...,-0.346075,-0.397462,-0.359658,-0.353801,-15.751804,-3.371562,0.983931,-4.565173,-4.836887,1.774853
std,0.146096,0.146096,0.146096,0.146791,0.142545,0.179352,1.418782,0.254960,1.273279,0.641606,...,0.280134,0.126892,0.153614,0.139140,2.467557,3.946629,2.690274,3.698768,8.451295,2.304695
min,-0.761689,-0.761689,-0.761689,-0.761689,-0.714635,-0.640921,-0.477598,-0.640921,-0.628132,-0.394224,...,-0.669796,-0.642778,-0.640921,-0.583553,-16.609640,-16.609640,-7.851749,-16.609640,-16.609640,-5.851749
25%,-0.583496,-0.583496,-0.583496,-0.583267,-0.569612,-0.492509,-0.291160,-0.434234,-0.307895,-0.012635,...,-0.443544,-0.521740,-0.456708,-0.434773,-16.609640,-4.730871,-0.570596,-5.899473,-16.609640,0.305353
50%,-0.444994,-0.444994,-0.444994,-0.444953,-0.441626,-0.361836,-0.162085,-0.307897,-0.184234,0.117315,...,-0.345407,-0.357853,-0.349805,-0.347298,-16.609640,-3.217481,1.155674,-4.527594,0.419117,1.577744
75%,-0.345154,-0.345154,-0.345154,-0.345154,-0.340478,-0.297026,0.301862,-0.216748,-0.061261,0.438901,...,-0.289041,-0.306664,-0.278969,-0.307569,-16.609640,-0.900504,2.266508,-2.949161,0.979063,2.822667
max,-0.185695,-0.185695,-0.185695,-0.185695,-0.169375,0.293136,5.688904,0.630576,6.708888,3.197611,...,1.486476,-0.185695,0.188576,0.261997,-7.679480,3.542786,7.553182,1.584963,5.828665,7.749900


In [44]:
df_lib.head()

,Contigs_ID,Similarity_label,X15,X16,X17,X18,X19,X20,X21,X22,...,X.33,X.34,X.35,dens15to18,dens20to22,dens25to29,ratiosi_pi,ratio_si,dens18to35,class
0,bN_SRR1174242_Contig101_100,viral,-0.624891,-0.624891,-0.624891,-0.624891,-0.624891,-0.433537,0.415597,-0.337860,...,-0.266102,-0.397658,-0.361779,-16.60964,-1.111361,1.563739,-2.675100,0.130704,2.253269,viral
1,bN_SRR1174242_Contig166_165,viral,-0.583553,-0.583553,-0.583553,-0.583553,-0.583553,-0.368087,-0.368087,-0.583553,...,-0.583553,-0.583553,-0.583553,-16.60964,-5.930737,-1.360882,-4.569856,0.000000,-0.897314,eve
2,bN_SRR1174242_Contig199_198,viral,-0.490232,-0.490232,-0.490232,-0.490232,-0.485109,-0.420564,-0.217706,-0.293180,...,-0.438664,-0.419881,-0.426028,-16.60964,-0.458279,4.086791,-4.545070,0.959460,4.387709,eve
3,bN_SRR1174242_Contig219_218,viral,-0.555396,-0.555396,-0.555396,-0.555396,-0.555396,-0.059414,0.312572,-0.431401,...,-0.555396,0.188576,-0.307405,-16.60964,-3.675565,-0.636037,-3.039528,1.000000,-0.124391,viral
4,bN_SRR1174242_Contig22_21,viral,-0.493885,-0.493885,-0.493885,-0.493885,-0.466948,-0.364972,1.332058,-0.237984,...,-0.349580,-0.276465,-0.336111,-16.60964,0.462526,2.295433,-1.832906,1.431453,3.378366,viral


## Build Data Frame

#### Auxiliary methods

Masks

In [37]:

# File size masks
bytes_2_kb = lambda size_bytes: size_bytes / (1024)
bytes_2_mb = lambda size_bytes: size_bytes / (1024 * 1024)
bytes_2_gb = lambda size_bytes: size_bytes / (1024 * 1024 * 1024)

bytes_2_gb_str = lambda size_bytes: "{:.2f} GB".format(bytes_2_gb(size_bytes))
bytes_2_mb_str = lambda size_bytes: "{:.2f} MB".format(bytes_2_mb(size_bytes))
bytes_2_kb_str = lambda size_bytes: "{:.2f} KB".format(bytes_2_kb(size_bytes))

# Scientific notation masks
mask_thousands = lambda x: "{:.2f}".format(x / 1000)
mask_million = lambda x: "{:.2f}".format(x / 1000 / 1000)


Report rows building

In [63]:
def add_masks(df_row: pd.DataFrame) -> pd.DataFrame:

    # Input size masks
    if df_row[col_input].values != None:
        df_row[f'{col_input} [GB]'] = df_row[col_input].apply(bytes_2_gb_str)
        df_row[f'{col_input} [MB]'] = df_row[col_input].apply(bytes_2_mb_str)
        df_row[f'{col_input} [KB]'] = df_row[col_input].apply(bytes_2_kb_str)

    # Output size masks
    if df_row[col_output].values != None:
        df_row[f'{col_output} [GB]'] = df_row[col_output].apply(bytes_2_gb_str)
        df_row[f'{col_output} [MB]'] = df_row[col_output].apply(bytes_2_mb_str)
        df_row[f'{col_output} [KB]'] = df_row[col_output].apply(bytes_2_kb_str)

    # Big numbers scientific notation
    if df_row[col_reads].values != None:
        df_row[f'{col_reads} [×10^3]'] = df_row[col_reads].apply(mask_thousands)
        df_row[f'{col_reads} [×10^6]'] = df_row[col_reads].apply(mask_million)

    return df_row

def get_similarity_class_count(df_lib: pd.DataFrame, label: str) -> int:
    try:
        similarity_class_group = df_lib.groupby(col_label_similarity)[col_label_similarity].value_counts()
        return int(similarity_class_group[label].values)
    except KeyError:
        return 0
    except Exception as err:
        raise err

def get_row(df_lib: pd.DataFrame, summary_lib: dict) -> pd.DataFrame:

    class_match = lambda label, class_: \
        df_lib[ (df_lib[col_label_similarity] == label) & (df_lib[col_label_class] == class_)]

    df_row = pd.DataFrame(
        {

            # 'Batch', 'Server',
            'Batch': summary_lib.get('batch_name'),
            'Server': summary_lib.get('server_name'),

            # 'ID', 'Lib prefix'
            'ID': summary_lib.get('exec_id'),
            'Lib prefix': summary_lib.get('lib_prefix'),

            # 'Host Reference'
            'Host Reference': summary_lib.get('path_ref'),

            # 'Execution Time', 'Execution path', '(range) Init. Size', '(range) End Size', 'Degradation Search'
            'Execution Time': summary_lib.get('exec_time'),
            'Execution path': summary_lib.get('path_exec_dir'),
            '(range) Init. Size': summary_lib.get('si'),
            '(range) End Size': summary_lib.get('se'),
            'Degradation Search': summary_lib.get('is_degrad_search') != 'TRUE',

            # 'Input file', 'Input Size', 'Output Size',
            'Input file': re.search(r'[^/]*$', summary_lib.get('path_input')).group(),
            col_input: summary_lib.get('input_size'),
            col_output: summary_lib.get('output_size'),

            # 'Reads', 'Reads Unmapped (host)', 'Reads Unmapped (bacteria)', 'Reads Preprocessed',
            col_reads: summary_lib.get('n_read_total'),
            'Reads Unmapped (host)': summary_lib.get('n_read_unmap_host'),
            'Reads Unmapped (bacteria)': summary_lib.get('n_read_unmap_bacter'),
            'Reads Preprocessed': summary_lib.get('n_read_preprocess'),

            # 'Contigs', 'Contigs >= 200nt'
            'Contigs': summary_lib.get('n_contig_total'),
            'Contigs >= 200nt': summary_lib.get('n_contig_gt200'),
            'Contigs >= 200nt (viral)': summary_lib.get('n_contig_all_viral'),
            'Contigs >= 200nt (non viral)': summary_lib.get('n_contig_all_non_viral'),
            'Contigs >= 200nt (no hits)': summary_lib.get('n_contig_all_nohit'),

            # 'Contig (blastn) Hits', 'Contig (blastn) No Hits', 'Contig (blastn) Viral', 'Contig (blastn) non viral',
            'Contig (blastn) Hits': summary_lib.get('n_contig_bn_hit'),
            'Contig (blastn) Viral': summary_lib.get('n_contig_bn_viral'),
            'Contig (blastn) non viral': summary_lib.get('n_contig_bn_non_viral'),
            'Contig (blastn) No Hits': summary_lib.get('n_contig_bn_nohit'),

            # 'Contig (diamond) Hits', 'Contig (diamond) No Hits', 'Contig (diamond) Viral', 'Contig (diamond) non viral',
            'Contig (diamond) Hits': summary_lib.get('n_contig_dmnd_hits'),
            'Contig (diamond) Viral': summary_lib.get('n_contig_dmnd_viral'),
            'Contig (diamond) non viral': summary_lib.get('n_contig_dmnd_non_viral'),
            'Contig (diamond) No Hits': summary_lib.get('n_contig_dmnd_nohit'),

            # 'Viral', 'non viral', 'nohit',
            # 'Viral / Viral', 'Viral / EVE',
            # 'non viral / Viral', 'non viral / EVE',
            # 'No Hit / Viral', 'No Hit / EVE',
            
        },
        index=[0]
    )

    df_row['Degradation Search'] = df_row['Degradation Search'].astype(bool)

    if not df_lib.empty:
        df_row['Viral'] = get_similarity_class_count(df_lib=df_lib, label='viral'),
        df_row['Non Viral'] = get_similarity_class_count(df_lib=df_lib, label='nonviral'),
        df_row['nohit'] = get_similarity_class_count(df_lib=df_lib, label='nohit'),
        df_row['Viral / Viral'] = class_match('viral', 'viral').shape[0],
        df_row['Viral / EVE'] = class_match('viral', 'eve').shape[0],
        df_row['Non Viral / Viral'] = class_match('nonviral', 'viral').shape[0],
        df_row['Non Viral / EVE'] = class_match('nonviral', 'eve').shape[0],
        df_row['No Hit / Viral'] = class_match('nohit', 'viral').shape[0],
        df_row['No Hit / EVE'] = class_match('nohit', 'eve').shape[0],

    return add_masks(df_row=df_row)


#### Gather executions

In [88]:
df_report = pd.DataFrame(
    columns = [
        'Batch', 'Server',
        'ID', 'Lib prefix',
        'Host Reference',
        'Execution Time', 'Execution path', '(range) Init. Size', '(range) End Size', 'Degradation Search',
        'Input file', 'Input Size', 'Output Size',
        'Reads', 'Reads Unmapped (host)', 'Reads Unmapped (bacteria)', 'Reads Preprocessed',
        'Contigs', 'Contigs >= 200nt', 'Contigs >= 200nt (viral)', 'Contigs >= 200nt (non viral)', 'Contigs >= 200nt (no hits)',
        'Contig (blastn) Hits', 'Contig (blastn) No Hits', 'Contig (blastn) Viral', 'Contig (blastn) non viral',
        'Contig (diamond) Hits', 'Contig (diamond) No Hits', 'Contig (diamond) Viral', 'Contig (diamond) non viral',
        'Viral', 'Non Viral', 'nohit',
        'Viral / Viral', 'Viral / EVE',
        'Non Viral / Viral', 'Non Viral / EVE',
        'No Hit / Viral', 'No Hit / EVE',
    ]
)

df_report['Degradation Search'] = df_report['Degradation Search'].astype(bool)

initial_columns = df_report.columns
print(f'initial_columns: {len(initial_columns)}')

initial_columns: 39


In [89]:


# Filter out subdirectories from the list of items
run_dirs = []
for dir_item in os.listdir(path_root):
    path_run = os.path.join(path_root, dir_item)
    if os.path.isdir(path_run):
        run_dirs.append(path_run)

# Loop trough each execution
for path_run in run_dirs:

    # Get execution result files
    path_csv_classified = ''
    path_summary = ''

    for dir_item in os.listdir(path_run):
        path_file = os.path.join(path_run, dir_item)

        if path_file.endswith(suffix_summary):
            path_summary = path_file
        elif path_file.endswith(suffix_classification):
            path_csv_classified = path_file
    
    if not path_summary:
        continue

    # Build current lib report row
    with open(path_summary, 'r') as file:
        summary_lib = yaml.safe_load(file)

    ''' 
        TODO: 2023-08-25 - Make it filled dynamically
    ''' 
    if not summary_lib.get('server_name'):
        summary_lib['server_name'] = server_r2d2

    df_lib = pd.read_csv(path_csv_classified).drop('Unnamed: 0', axis=1) if path_csv_classified else pd.DataFrame()
    df_row = get_row(df_lib=df_lib, summary_lib=summary_lib)

    # Add this lib parsing to report
    df_report = pd.concat([df_report, df_row], ignore_index=True)


In [90]:
df_report.head()

,Batch,Server,ID,Lib prefix,Host Reference,Execution Time,Execution path,(range) Init. Size,(range) End Size,Degradation Search,...,No Hit / Viral,No Hit / EVE,Input Size [GB],Input Size [MB],Input Size [KB],Output Size [GB],Output Size [MB],Output Size [KB],Reads [×10^3],Reads [×10^6]
0,batch-15,r2d2,val-SRR1056333,SRR1056333,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,04:29:58:352,./runs/val-SRR1056333,15,35,True,...,NaN,NaN,0.00 GB,0.69 MB,705.00 KB,0.07 GB,70.23 MB,71916.22 KB,8268.19,8.27
1,batch-03,r2d2,val-SRR2182528,SRR2182528,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,04:15:02:303,./runs/val-SRR2182528,15,35,True,...,8,24,0.00 GB,1.48 MB,1512.94 KB,0.02 GB,17.80 MB,18227.28 KB,25001.33,25.00
2,batch-05,aw1,val-ERR186228,ERR186228,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,11:52:03:529,./runs/val-ERR186228,15,35,True,...,1,7,0.00 GB,1.23 MB,1257.60 KB,0.00 GB,4.07 MB,4172.50 KB,12939.51,12.94
3,batch-03,r2d2,val-SRR2182437,SRR2182437,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,05:41:27:203,./runs/val-SRR2182437,15,35,True,...,5,117,0.00 GB,1.21 MB,1241.05 KB,0.01 GB,9.05 MB,9269.37 KB,20134.60,20.13
4,batch-13,r2d2,val-SRR5163620,SRR5163620,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,02:29:42:263,./runs/val-SRR5163620,15,35,True,...,0,2,0.00 GB,1.74 MB,1780.70 KB,0.01 GB,5.46 MB,5586.81 KB,18924.44,18.92


In [91]:
final_columns = df_report.columns
print(f'final_columns: {len(final_columns)}')

new_columns = [c for c in final_columns if c not in initial_columns]
new_columns

final_columns: 47


['Input Size [GB]',
 'Input Size [MB]',
 'Input Size [KB]',
 'Output Size [GB]',
 'Output Size [MB]',
 'Output Size [KB]',
 'Reads [×10^3]',
 'Reads [×10^6]']

In [92]:
sorted(df_report['Batch'].unique())

['batch-01',
 'batch-02',
 'batch-03',
 'batch-04',
 'batch-05',
 'batch-06',
 'batch-07',
 'batch-09',
 'batch-10',
 'batch-11',
 'batch-12',
 'batch-13',
 'batch-14',
 'batch-15']

In [93]:
df_report.groupby('Batch')['Batch'].value_counts()

Batch     Batch   
batch-01  batch-01    14
batch-02  batch-02    20
batch-03  batch-03    20
batch-04  batch-04     3
batch-05  batch-05     4
batch-06  batch-06     6
batch-07  batch-07     3
batch-09  batch-09     4
batch-10  batch-10     5
batch-11  batch-11     1
batch-12  batch-12     5
batch-13  batch-13    10
batch-14  batch-14     5
batch-15  batch-15     3
Name: Batch, dtype: int64

In [94]:
df_report.groupby('Batch')['Batch'].value_counts().sum()

103

#### Check on empty rows

In [95]:
df_report.describe()

,Batch,Server,ID,Lib prefix,Host Reference,Execution Time,Execution path,(range) Init. Size,(range) End Size,Degradation Search,...,No Hit / Viral,No Hit / EVE,Input Size [GB],Input Size [MB],Input Size [KB],Output Size [GB],Output Size [MB],Output Size [KB],Reads [×10^3],Reads [×10^6]
count,103,103,103,103,103,103,103,103,103,103,...,88,88,97,97,97,103,103,103,102,102
unique,14,3,103,103,1,103,103,1,1,1,...,12,38,1,90,96,8,96,103,101,101
top,batch-03,r2d2,val-SRR1056333,SRR1056333,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,04:29:58:352,./runs/val-SRR1056333,15,35,True,...,0,0,0.00 GB,2.21 MB,1345.50 KB,0.01 GB,4.07 MB,71916.22 KB,13283.48,13.28
freq,20,89,1,1,103,1,1,103,103,103,...,41,16,97,2,2,44,3,1,2,2


In [96]:
df_report.shape

(103, 47)

In [97]:
df_report.isna().sum()

Batch                            0
Server                           0
ID                               0
Lib prefix                       0
Host Reference                   0
Execution Time                   0
Execution path                   0
(range) Init. Size               0
(range) End Size                 0
Degradation Search               0
Input file                       0
Input Size                       6
Output Size                      0
Reads                            1
Reads Unmapped (host)            1
Reads Unmapped (bacteria)        1
Reads Preprocessed               1
Contigs                          1
Contigs >= 200nt                 2
Contigs >= 200nt (viral)        23
Contigs >= 200nt (non viral)    23
Contigs >= 200nt (no hits)      23
Contig (blastn) Hits            23
Contig (blastn) No Hits         23
Contig (blastn) Viral           23
Contig (blastn) non viral       23
Contig (diamond) Hits           23
Contig (diamond) No Hits        23
Contig (diamond) Vir

In [99]:
df_report[ df_report['Reads'].isna() | df_report['Batch'].isna()]

,Batch,Server,ID,Lib prefix,Host Reference,Execution Time,Execution path,(range) Init. Size,(range) End Size,Degradation Search,...,No Hit / Viral,No Hit / EVE,Input Size [GB],Input Size [MB],Input Size [KB],Output Size [GB],Output Size [MB],Output Size [KB],Reads [×10^3],Reads [×10^6]
44,batch-01,r2d2,val-SRR10662835,SRR10662835,/small-rna-metavir/asset/refs/ref_hosts/refs_A...,00:00:06:445,./runs/val-SRR10662835,15,35,True,...,0,0,0.00 GB,1.93 MB,1978.21 KB,0.01 GB,7.23 MB,7404.04 KB,NaN,NaN


In [100]:
df_report.drop(df_report[ df_report['ID'] == 'val-SRR10662835' ].index, inplace=True)

In [101]:
df_report[ df_report['Reads'].isna() | df_report['Batch'].isna()]

,Batch,Server,ID,Lib prefix,Host Reference,Execution Time,Execution path,(range) Init. Size,(range) End Size,Degradation Search,...,No Hit / Viral,No Hit / EVE,Input Size [GB],Input Size [MB],Input Size [KB],Output Size [GB],Output Size [MB],Output Size [KB],Reads [×10^3],Reads [×10^6]


In [102]:
df_report.shape

(102, 47)

#### Export

In [103]:
df_report.to_csv(path_export, index=False)